# L03 · SFT·오프폴리시·온폴리시

## Goal

**예상 시간:** 40분 · **경로:** fast, full

- 학습 state의 생성자를 구분한다
- SFT와 KD를 공정 비교한다
- response budget을 감사한다

### 현재 위치: L02 → **L03** → L04

```text
Prompt/Data -> state source -> ... -> L03 -> ... -> fair evaluation
```

Alt text: The course map highlights L03 between its prerequisite and next lesson; every method remains connected to the same evaluation stage.

## Setup

In [1]:
LESSON_ID = "L03"
from pathlib import Path
import sys
import torch

repo_root = Path.cwd()
if not (repo_root / "src").exists():
    repo_root = Path.cwd().parents[1]
sys.path.insert(0, str(repo_root / "src"))

import opd_study
from opd_study.device import resolve_device
from opd_study.utils import seed_everything

seed_everything(42)
device_report = resolve_device("cpu")
print({"lesson": LESSON_ID, "opd_study": opd_study.__version__,
       "torch": torch.__version__, "device": device_report.selected,
       "profile": "toy", "network": "not required"})

{'lesson': 'L03', 'opd_study': '0.1.0.dev0', 'torch': '2.13.0', 'device': 'cpu', 'profile': 'toy', 'network': 'not required'}


## Steps

### 1/3 · 8–12 min

SFT와 off-policy KD는 target 형식은 다르지만 고정된 demonstration prefix에서 학습한다. 공정 비교는 같은 student 초기화와 response-token budget부터 시작한다.

그림 대체 설명: 출력의 label과 숫자는 색 없이도 읽을 수 있다.

### 핵심 원리

학습 state 분포를 기호로 쓰면 SFT/KD는 대체로 `s ~ d_data`, OPD는 `s ~ d_student`다. 모델이 추론 중 스스로 만든 오류 prefix는 `d_data`에 거의 없을 수 있다. OPD는 이 covariate shift를 직접 보지만, student가 너무 나쁜 state만 방문하면 오히려 teacher 신호가 불안정해질 수 있다.

공정 비교는 같은 초기 weight, prompt subset, optimizer, learning rate, response-token 수, 평가 split을 고정한다. optimizer step만 맞추면 길이가 긴 방법이 더 많은 token을 볼 수 있으므로 token budget도 별도로 기록한다.

### 실제 구현: 왜 이렇게 만들었나

`TrajectoryBatch`가 token IDs, attention/response mask, prompt length와 rollout snapshot을 한 계약으로 묶는다. SFT와 KD는 동일 batch의 동일 response target 수를 쓰되, SFT는 token ID, KD는 detached teacher logits을 읽는다.

실제 코드: [`types.py`](../../src/opd_study/types.py), [`core.py`](../../src/opd_study/training/core.py).

In [2]:
import inspect
from opd_study.algorithms import supervised_fine_tuning_loss, off_policy_kd_loss

objects_to_show = (supervised_fine_tuning_loss, off_policy_kd_loss,)
for object_to_show in objects_to_show:
    source_lines = inspect.getsource(object_to_show).splitlines()
    print(f"\n# {object_to_show.__module__}.{object_to_show.__qualname__}")
    print("\n".join(source_lines[:80]))
    if len(source_lines) > 80:
        print(f"... {len(source_lines) - 80} more lines; open the linked source file")


# opd_study.algorithms.losses.supervised_fine_tuning_loss
def supervised_fine_tuning_loss(
    student_logits: Tensor,
    trajectories: TrajectoryBatch,
) -> LossOutput:
    """Hard-label next-token cross entropy on demonstration response tokens."""

    shifted_logits, target_ids, _, prediction_mask = shifted_causal_tensors(
        student_logits,
        trajectories.token_ids,
        trajectories.attention_mask,
        trajectories.response_mask,
    )
    shifted_loss = cross_entropy_from_logits(shifted_logits, target_ids)
    loss = masked_mean(shifted_loss, prediction_mask)
    token_loss, effective_mask = _restore_token_alignment(
        shifted_loss, prediction_mask, trajectories.token_ids.shape[1]
    )
    return LossOutput(
        loss=loss,
        token_loss=token_loss,
        effective_mask=effective_mask,
        metrics=_metrics(loss, effective_mask, prefix="sft"),
    )

# opd_study.algorithms.losses.off_policy_kd_loss
def off_policy_kd_loss(
    student_logits

### 다른 선택지는 없나?

state source를 batch 단위, example 단위, token/turn 단위로 섞을 수 있다. 이 mini runtime은 재현성과 설명을 위해 batch 단위로 고른다. 더 세밀한 혼합은 분산을 줄일 수 있지만 trajectory provenance가 복잡해진다.

### 2/3 · 실행하고 관찰하기

실행 전 예측: L03의 첫 출력에서 가장 먼저 확인해야 할 invariant는 무엇일까? 한 문장으로 적고 실행한다.

In [3]:
import copy
from opd_study.algorithms import off_policy_kd_loss, score_teacher, supervised_fine_tuning_loss
from opd_study.data import CharacterTokenizer, collate_examples, generate_tiny_arithmetic
from opd_study.models import TinyCausalLM, TinyTransformerConfig

tokenizer = CharacterTokenizer(); splits = generate_tiny_arithmetic(train_rows=8, validation_rows=2, test_rows=2)
batch = collate_examples(splits.train[:2], tokenizer)
config = TinyTransformerConfig(vocab_size=tokenizer.vocab_size, number_of_layers=1,
    hidden_size=32, number_of_heads=4, feed_forward_size=64)
initial = TinyCausalLM(config); teacher = TinyCausalLM(config)
sft_student = copy.deepcopy(initial); kd_student = copy.deepcopy(initial)
teacher_signals = score_teacher(teacher, batch)
sft_output = supervised_fine_tuning_loss(sft_student(batch.token_ids, batch.attention_mask), batch)
kd_output = off_policy_kd_loss(kd_student(batch.token_ids, batch.attention_mask), batch, teacher_signals)
print("same response targets:", int(sft_output.effective_mask.sum()), int(kd_output.effective_mask.sum()))

same response targets: 68 68


In [4]:
from opd_study.utils import model_state_hash

print("initial hashes equal:", model_state_hash(sft_student) == model_state_hash(kd_student))
print("SFT reads hard target IDs; off-policy KD reads teacher distributions on the same fixed prefixes.")

initial hashes equal: True
SFT reads hard target IDs; off-policy KD reads teacher distributions on the same fixed prefixes.


## Checks

In [5]:
assert model_state_hash(sft_student) == model_state_hash(kd_student)
assert int(sft_output.effective_mask.sum()) == int(kd_output.effective_mask.sum())
assert not teacher_signals.logits.requires_grad
print("check passed: initialization, state source, token budget, and teacher detach are auditable")

check passed: initialization, state source, token budget, and teacher detach are auditable


**연습 (7분):** SFT batch와 KD batch의 response mask를 하나씩 출력하고 prompt 길이를 두 배로 늘려도 effective token budget이 변하지 않는 assertion을 추가하라.

<details><summary>확인 기준</summary>prompt 위치는 모두 false이고 response target 수만 budget에 들어가야 한다.</details>

## 내가 자주 틀리는 것

### M1 — target이 같으면 state도 같다고 보기

- 틀린 형태: SFT와 KD가 같은 answer를 쓰니 같은 학습이라고 한다.
- 왜 틀렸나: hard ID와 teacher distribution은 정보량이 다르다.
- 고친 형태: state source와 target representation을 각각 표시한다.
- 관련 검사: `test_teacher_is_frozen_and_student_updates`

### M2 — optimizer step만 공정성으로 보고하기

- 틀린 형태: 길이가 달라도 step 수만 맞춘다.
- 왜 틀렸나: 처리한 response token 수가 달라질 수 있다.
- 고친 형태: initial hash, split, step과 token budget을 모두 비교한다.
- 관련 검사: `test_demo_writes_all_learner_facing_artifacts`

## 60초 요약

1. 학습 state의 생성자를 구분한다
2. SFT와 KD를 공정 비교한다
3. response budget을 감사한다

## Next Steps

다음 노트북으로 가기 전, 위 assertion을 다시 실행하고 틀린 예측 한 줄을 남긴다.

### Sources

- [`gkd`](https://arxiv.org/abs/2306.13649v3) · `2306.13649v3` · license `CC-BY-4.0` · [audited manifest](../../docs/sources.yml)